In [6]:
import re
import os
import json
from collections import Counter

def clean_hindi_text(text):
    """
    Strips out English letters, numbers, and punctuation.
    Keeps ONLY Devanagari characters and spaces.
    """
    cleaned = re.sub(r'[^\u0900-\u097F\s]', '', text)
    return ' '.join(cleaned.split())

def build_and_save_lexicon_full(file_path, save_path, min_freq=5):
    """
    Builds the hindi word dictionary.
    """
    if not os.path.exists(file_path):
        print(f"Error: Cannot find the dataset at '{file_path}'.")
        return None

    word_frequencies = Counter()
    lines_processed = 0
    
    print(f"Scanning the entire corpus: {file_path}...")
    print("This will take a few minutes...\n")
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            cleaned_line = clean_hindi_text(line)
            
            if cleaned_line:
                word_frequencies.update(cleaned_line.split())
                lines_processed += 1
                
            # Progress tracker (updated to 1 million)
            if lines_processed % 1000000 == 0:
                print(f"Processed {lines_processed:,} sentences...")

    # Filter out rare words (typos in the clean corpus itself)
    filtered_dict = {word: count for word, count in word_frequencies.items() if count >= min_freq}
    
    print(f"\n--- FULL EXTRACTION COMPLETE ---")
    print(f"Total lines processed: {lines_processed:,}")
    print(f"Total highly reliable words found: {len(filtered_dict):,}")
    
    # Save the dictionary to a new JSON file
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(filtered_dict, f, ensure_ascii=False, indent=4)
        
    print(f"Full lexicon successfully saved to: {save_path}")
    
    return filtered_dict


# RUN THE FULL EXTRACTION


CORPUS_PATH = "data/monolingual.hi" 
SAVE_PATH = "data/hindi_lexicon_full.json"

valid_dictionary_full = build_and_save_lexicon_full(
    file_path=CORPUS_PATH, 
    save_path=SAVE_PATH, 
    min_freq=5
)

# Verify it worked by printing the top 15 words
if valid_dictionary_full:
    print("\nTop 15 most frequent words across the entire corpus:")
    for word, count in sorted(valid_dictionary_full.items(), key=lambda x: x[1], reverse=True)[:15]:
        print(f"{word}: {count:,}")

Scanning the entire corpus: data/monolingual.hi...
This will take a few minutes...

Processed 1,000,000 sentences...
Processed 2,000,000 sentences...
Processed 3,000,000 sentences...
Processed 4,000,000 sentences...
Processed 5,000,000 sentences...
Processed 6,000,000 sentences...
Processed 7,000,000 sentences...
Processed 8,000,000 sentences...
Processed 9,000,000 sentences...
Processed 10,000,000 sentences...
Processed 11,000,000 sentences...
Processed 12,000,000 sentences...
Processed 13,000,000 sentences...
Processed 14,000,000 sentences...
Processed 15,000,000 sentences...
Processed 16,000,000 sentences...
Processed 17,000,000 sentences...
Processed 18,000,000 sentences...
Processed 19,000,000 sentences...
Processed 20,000,000 sentences...
Processed 21,000,000 sentences...
Processed 22,000,000 sentences...
Processed 23,000,000 sentences...
Processed 24,000,000 sentences...
Processed 25,000,000 sentences...
Processed 26,000,000 sentences...
Processed 27,000,000 sentences...
Process

In [8]:
# CONTEXTUAL WHITELISTING (NER)


from transformers import pipeline
import torch

# Load multilingual NER model
print("Downloading Babelscape Multilingual NER model...")
ner_tagger = pipeline( # type: ignore
    "ner", 
    model="Babelscape/wikineural-multilingual-ner", 
    aggregation_strategy="simple", 
)
print("NER Model loaded successfully!\n")

def get_ner_whitelist(sentence):
    """
    Scans a sentence and returns a list of words that should NOT be spell-checked 
    (Persons, Locations, Organizations, etc.).
    """
    entities = ner_tagger(sentence)
    
    whitelist = []
    for ent in entities:
        # Some models use subwords, this cleans it up
        clean_word = ent['word'].replace('#', '').strip()
        
        # Only add valid, non-empty words
        if clean_word:
            whitelist.append({
                "word": clean_word,
                "type": ent['entity_group'], 
                "score": round(ent['score'], 4)
            })
        
    return whitelist


# TEST THE WHITELIST EXTRACTOR

test_sentence = "सुशांत और महेंद्र सिंह धोनी कल दिल्ली के ताज होटल में गए थे।"
print(f"Original Sentence: {test_sentence}\n")

protected_entities = get_ner_whitelist(test_sentence)

print("Words Protected from the Spell Checker:")
for item in protected_entities:
    print(f" - {item['word']} ({item['type']}) | Confidence: {item['score']}")

bypass_list = [item['word'] for item in protected_entities]
print(f"\nFinal Whitelist Array: {bypass_list}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER Model loaded successfully!

Original Sentence: सुशांत और महेंद्र सिंह धोनी कल दिल्ली के ताज होटल में गए थे।

Words Protected from the Spell Checker:
 - शांत (PER) | Confidence: 0.6502000093460083
 - महेंद्र सिंह धोनी (PER) | Confidence: 0.8935999870300293
 - दिल्ली (LOC) | Confidence: 0.9936000108718872

Final Whitelist Array: ['शांत', 'महेंद्र सिंह धोनी', 'दिल्ली']


In [9]:
# SYNTHETIC ERROR GENERATION


import random

CONFUSION_MATRIX = {
    'श': ['स', 'ष'],
    'स': ['श', 'ष'],
    'ष': ['श', 'स'],
    'ड़': ['ड'],
    'ड': ['ड़'],
    'ढ़': ['ढ'],
    'ढ': ['ढ़'],
    'ज': ['ज़'],
    'ज़': ['ज'],
    'फ': ['फ़'],
    'फ़': ['फ'],
    'इ': ['ई'],
    'ई': ['इ'],
    'उ': ['ऊ'],
    'ऊ': ['उ'],
    'ि': ['ी'], 
    'ी': ['ि'],
    'ु': ['ू'], 
    'ू': ['ु'],
    'े': ['ै'],
    'ै': ['े']
}

def generate_hindi_typo(word):
    """
    Takes a correct Hindi word and intentionally injects a realistic human error.
    """
    # If the word is too short, mangling it usually destroys it completely
    if len(word) < 3:
        return word 
        
    # Randomly select which type of error to simulate
    error_type = random.choice(["phonetic", "drop_char", "swap_char"])
    word_list = list(word)
    
    if error_type == "phonetic":
        # Find all characters in the word that exist in our confusion matrix
        possible_swaps = [i for i, char in enumerate(word_list) if char in CONFUSION_MATRIX]
        
        if possible_swaps:
            idx = random.choice(possible_swaps)
            # Swap the correct character with a common mistake
            word_list[idx] = random.choice(CONFUSION_MATRIX[word_list[idx]])
        else:
            # Fallback to character dropping if no phonetic swaps are available
            error_type = "drop_char" 
            
    if error_type == "drop_char":
        # Drop a random character (usually a matra or middle consonant)
        idx = random.randint(1, len(word_list) - 1)
        word_list.pop(idx)
        
    elif error_type == "swap_char":
        # Swap two adjacent characters to simulate a typing slip
        idx = random.randint(0, len(word_list) - 2)
        word_list[idx], word_list[idx+1] = word_list[idx+1], word_list[idx]
        
    return "".join(word_list)


# TEST THE GENERATOR


clean_test_words = ["किताब", "सुशांत", "पढ़ाई", "जरूर", "वापस", "आशीर्वाद", "परीक्षा"]

print("Synthetic Error Generator Test:")
print("-" * 40)
for word in clean_test_words:
    typo = generate_hindi_typo(word)
    print(f"Clean: {word} -> Typo: {typo}")

Synthetic Error Generator Test:
----------------------------------------
Clean: किताब -> Typo: कितबा
Clean: सुशांत -> Typo: सुशां
Clean: पढ़ाई -> Typo: पढ़ाइ
Clean: जरूर -> Typo: ज़रूर
Clean: वापस -> Typo: वाप
Clean: आशीर्वाद -> Typo: आषीर्वाद
Clean: परीक्षा -> Typo: परीकष्ा


In [ ]:

# THE CORE MuRIL PIPELINE (FIXED)


import json
import string
from transformers import pipeline
import torch

# 1. Load the Lexicon from Phase 1
print("Loading the Phase 1 Dictionary into memory...")
try:
    with open('data/hindi_lexicon_full.json', 'r', encoding='utf-8') as f:
        hindi_dictionary = json.load(f)
    print(f"Lexicon loaded successfully! Total words: {len(hindi_dictionary):,}")
except FileNotFoundError:
    print("Error: hindi_lexicon_full.json not found. Did you run Phase 1?")
    hindi_dictionary = {}

# 2. Setup Device and Load MuRIL
active_device = 0 if torch.cuda.is_available() else -1
print("\nLoading the Google MuRIL Base model...")
muril_mlm = pipeline( # type: ignore
    "fill-mask", 
    model="google/muril-base-cased", 
    device=active_device
)
print("MuRIL loaded successfully!\n")

# 3. Define the Edit Distance Algorithm
def get_edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0 for _ in range(n + 1)] for _ in range(m + 1)]
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0: dp[i][j] = j
            elif j == 0: dp[i][j] = i
            elif word1[i-1] == word2[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i][j-1], dp[i-1][j], dp[i-1][j-1])
    return dp[m][n]

# 4. The Master Spell Checking Logic
def correct_hindi_sentence(sentence, min_freq=5000):
    protected_entities = get_ner_whitelist(sentence)
    bypass_list = [item['word'] for item in protected_entities]
    
    words = sentence.split()
    corrected_sentence = []
    corrections_log = []
    
    # Dynamically grab the official mask token from the model
    mask_token = muril_mlm.tokenizer.mask_token
    
    for i, word in enumerate(words):
        # FIX: Safely strip punctuation only from the outside edges
        clean_word = word.strip(string.punctuation + "।॥")
        
        # If it was just a lone punctuation mark, skip it
        if not clean_word:
            corrected_sentence.append(word)
            continue
        
        # Rule 1: Is it a Named Entity?
        if clean_word in bypass_list:
            corrected_sentence.append(word)
            continue
            
        # Rule 2: Is it a VALID, FREQUENT word in the Dictionary?
        word_freq = hindi_dictionary.get(clean_word, 0) if isinstance(hindi_dictionary, dict) else 0
        if clean_word in hindi_dictionary and word_freq > min_freq:
            corrected_sentence.append(word)
            continue
            
        # Rule 3: It failed both. It is a Typo! 
        masked_words = words.copy()
        
        # Replace the clean_word with the official mask_token
        masked_words[i] = word.replace(clean_word, mask_token)
        masked_sentence = " ".join(masked_words)
        
        # Failsafe: Force the mask if the replace logic missed an extreme edge case
        if mask_token not in masked_sentence:
            masked_words[i] = mask_token
            masked_sentence = " ".join(masked_words)
        
        predictions = muril_mlm(masked_sentence, top_k=1000)
        
        best_prediction = clean_word
        lowest_distance = float('inf')
        
        for pred in predictions:
            pred_word = pred['token_str'].replace('#', '').strip()
            if not pred_word: continue
                
            dist = get_edit_distance(clean_word, pred_word)
            if dist < lowest_distance:
                lowest_distance = dist
                best_prediction = pred_word
        
        corrections_log.append(f"Typo: '{clean_word}' -> Fixed: '{best_prediction}' (Edit Distance: {lowest_distance})")
        corrected_sentence.append(word.replace(clean_word, best_prediction))
        
    return " ".join(corrected_sentence), corrections_log

# ==========================================
# TEST THE MASTER PIPELINE
# ==========================================

test_typo_sentence = "सुशांत ने एक नई कीताब खरीदी।"

print("Running Master Pipeline...")
print(f"Input:  {test_typo_sentence}")

final_text, logs = correct_hindi_sentence(test_typo_sentence, min_freq=100)

print(f"Output: {final_text}\n")
print("Correction Log:")
if not logs:
    print(" - No corrections needed (or everything bypassed).")
for log in logs:
    print(f" - {log}")

Loading the Phase 1 Dictionary into memory...
Lexicon loaded successfully! Total words: 779,458

Loading the Google MuRIL Base model...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: google/muril-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored w

MuRIL loaded successfully!

Running Master Pipeline...
Input:  सुशांत ने एक नई कीताब खरीदी।
Output: सुशांत ने एक नई किताब खरीदी।

Correction Log:
 - Typo: 'कीताब' -> Fixed: 'किताब' (Edit Distance: 1)


In [11]:
# INTERFACE & DEPLOYMENT (GRADIO)

import gradio as gr

def spell_check_app(user_text):
    """
    The wrapper function that connects Gradio's inputs to your Phase 4 pipeline.
    """
    if not user_text.strip():
        return "", "Please enter some Hindi text."
        
    # Run the text through your master function
    final_text, logs = correct_hindi_sentence(user_text, min_freq=100)
    
    # Format the logs nicely for the UI
    if not logs:
        log_output = "[Success] No spelling errors found (or all words were bypassed)."
    else:
        log_output = "\n".join([f"-> {log}" for log in logs])
        
    return final_text, log_output

# Build the Web Interface
print("Launching the Web Interface...")
demo = gr.Interface(
    fn=spell_check_app,
    inputs=gr.Textbox(
        lines=4, 
        placeholder="यहाँ हिंदी वाक्य लिखें... (e.g., सुशांत ने एक नई कीताब खरीदी।)", 
        label="Original Hindi Text"
    ),
    outputs=[
        gr.Textbox(lines=4, label="Corrected Text"),
        gr.Textbox(lines=4, label="Correction Log")
    ],
    title="AI-Powered Hindi Spell Checker",
    description="This tool uses Google's MuRIL transformer, Levenshtein edit distance, and an NER model to contextually correct Hindi spelling mistakes while protecting names and locations.",
    flagging_mode="never" 
)

# Launch the app directly inside the notebook and create a local web link
demo.launch(share=False, inbrowser=True)

Launching the Web Interface...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# MASSIVE PERFORMANCE EVALUATION


import time
import random
from tqdm.auto import tqdm

BASE_CLEAN_SENTENCES = [
    "सुशांत ने एक नई किताब खरीदी।",
    "कल मेरी गणित की परीक्षा है।",
    "वह पिछले दो दिनों से बीमार है।",
    "महेंद्र सिंह धोनी भारत के महान कप्तान थे।",
    "बड़ों का आशीर्वाद हमेशा लेना चाहिए।",
    "यह एक बहुत ही सुंदर कविता है।",
    "हमें रोज़ सुबह व्यायाम करना चाहिए।",
    "विज्ञान के क्षेत्र में भारत आगे बढ़ रहा है।",
    "ताजमहल दुनिया के सात अजूबों में से एक है।",
    "सफलता के लिए कठिन परिश्रम ज़रूरी है।",
    "माँ ने आज स्वादिष्ट खिचड़ी बनाई है।",
    "बारिश होने पर सड़कें कीचड़ से भर जाती हैं।",
    "उसने अपनी पुरानी साइकिल बेच दी।",
    "हम सब मिलकर पार्क में खेलते हैं।",
    "डॉक्टर ने उसे तीन दिन का आराम करने को कहा।",
    "यह फूल बहुत सुगंधित है।",
    "राम ने अपने दोस्त को जन्मदिन की शुभकामनाएँ दीं।",
    "स्कूल में आज वार्षिक खेलकूद का आयोजन है।",
    "पेड़ों की छाँव में बैठना बहुत अच्छा लगता है।",
    "मेरे पिताजी हर सुबह समाचार पत्र पढ़ते हैं।",
    "वह बहुत मेहनती और ईमानदार लड़का है।",
    "गर्मियों की छुट्टियों में हम गाँव जाते हैं।",
    "चाँदनी रात में तारे बहुत चमकते हैं।",
    "मुझे संगीत सुनना बहुत पसंद है।",
    "उसने परीक्षा में प्रथम स्थान प्राप्त किया।",
    "हमारे देश का राष्ट्रीय पशु बाघ है।",
    "सूरज हर सुबह पूर्व दिशा से उगता है।",
    "बच्चों को समय पर सोना चाहिए।",
    "यह फिल्म बहुत प्रेरणादायक है।",
    "मेरी बहन रोज़ स्कूल पैदल जाती है।",
    "हवा में ठंडक बढ़ गई है।",
    "उसने नई डायरी में पहला पन्ना लिखा।",
    "हमने पिछले रविवार को पिकनिक मनाई।",
    "अच्छे स्वास्थ्य के लिए संतुलित आहार ज़रूरी है।",
    "वह अपने दादाजी के साथ चेस खेलता है।",
    "बारिश के कारण मैच रद्द हो गया।",
    "मेरे पास एक छोटा सा कुत्ता है।",
    "उसने अपनी बचत से मोबाइल खरीदा।",
    "स्कूल की छुट्टी में हम घूमने गए थे।",
    "यह कहानी बहुत रोचक है।",
    "माँ ने सब्ज़ी में ज्यादा मसाले डाल दिए।",
    "ट्रेन ठीक समय पर प्लेटफ़ॉर्म पर आई।",
    "उसके पास बहुत सुंदर हैंडबैग है।",
    "हम लोग हर रविवार को मंदिर जाते हैं।",
    "पढ़ाई के साथ खेलकूद भी महत्वपूर्ण है।",
    "वह बहुत जल्दी-जल्दी बोलता है।",
    "आज मौसम बहुत सुहावना है।",
    "मेरे दोस्त ने नया लैपटॉप लिया है।",
    "पक्षी सुबह-सुबह चहचहाते हैं।",
    "उसने अपनी डायरी में सपने लिखे।",
    "हमारे गाँव में बहुत सारे आम के पेड़ हैं।",
    "बिजली चली गई तो मोमबत्ती जलाई।",
    "वह रोज़ सुबह योग करता है।",
    "इस किताब में बहुत सारी नई बातें हैं।",
    "मेरी मम्मी बहुत अच्छा खाना बनाती हैं।",
    "स्कूल में आज विज्ञान प्रदर्शनी लगी है।",
    "उसने अपने भाई को जन्मदिन का तोहफा दिया।",
    "सड़क पर बहुत तेज़ ट्रैफिक था।",
    "हमने समुद्र तट पर सूर्यास्त देखा।",
    "वह बहुत शांत और समझदार लड़की है।",
    "पिता जी ने नई कार खरीद ली है।",
    "बच्चे पार्क में झूले झूल रहे हैं।",
    "मुझे कॉफी से ज्यादा चाय पसंद है।",
    "उसने अपनी पुरानी नोटबुक फाड़ दी।",
    "आज स्कूल में विशेष सभा हुई।",
    "हमारे घर के पास एक बड़ा तालाब है।",
    "वह बहुत अच्छी कविताएँ लिखती है।",
    "बारिश में भीगना मुझे अच्छा लगता है।",
    "मेरे चचेरे भाई ने नया फोन लिया है।",
    "सूरज ढलने के बाद ठंड बढ़ जाती है।",
    "उसने अपनी परीक्षा अच्छे नंबरों से पास की।",
    "हम लोग हर साल दशहरा मना रहे हैं।",
    "यहाँ का पानी बहुत मीठा है।",
    "माँ ने आज पनीर की सब्जी बनाई।",
    "वह अपने दोस्तों के साथ क्रिकेट खेलता है।",
    "स्कूल की छुट्टी में किताबें पढ़ता हूँ।",
    "उसने अपनी माँ को सरप्राइज़ दिया।",
    "हमारे शहर में बहुत सारे पार्क हैं।",
    "पेड़ से आम तोड़कर खाए।",
    "वह बहुत जल्दी उठ जाता है।",
    "मेरे पास एक सुंदर डायरी है।",
    "आज का दिन बहुत व्यस्त था।",
    "उसने नई साड़ी पहनी है।",
    "हम सब मिलकर गाना गा रहे थे।",
    "बारिश के बाद इंद्रधनुष दिखाई दिया।",
    "मेरी छोटी बहन बहुत शरारती है।",
    "वह रोज़ स्कूल की बस से आता है।",
    "इस गाने की धुन बहुत अच्छी है।",
    "हमने बाजार से ताज़े फल खरीदे।",
    "उसके पास बहुत सारे स्टिकर हैं।",
    "मुझे मैथ्स की प्रैक्टिस करनी है।",
    "वह बहुत अच्छी ड्राइंग बनाती है।",
    "आज रात को चाँद बहुत बड़ा दिख रहा है।",
    "हमारे पड़ोस में नया परिवार आया है।",
    "उसने अपनी कॉपी बहुत साफ रखी है।",
    "माँ ने सबको गरम-गरम पराठे दिए।",
    "स्कूल में आज नया टीचर आया।",
    "वह अपने पालतू बिल्ली से बहुत प्यार करता है।",
    "हम लोग हर शाम पार्क में टहलते हैं।",
    "उसने अपनी पुरानी जूती फेंक दी।",
    "यह कहानी मेरे दादाजी ने सुनाई थी।",
    "मुझे नीले रंग की टी-शर्ट पसंद है।",
    "वह बहुत तेज़ दौड़ता है।",
    "हमने रविवार को मूवी देखी।",
    "उसके पास बहुत सारे खिलौने हैं।",
    "आज का मौसम बहुत अच्छा है।",
    "मेरी मम्मी रोज़ सुबह पूजा करती हैं।",
    "वह अपने दोस्त के घर गया है।",
    "हम सब मिलकर केक काट रहे हैं।",
    "उसने नई पेन खरीदी है।",
    "स्कूल में आज स्पीच कॉम्पिटिशन है।",
    "मुझे गर्म दूध पीना अच्छा लगता है।",
    "वह बहुत अच्छी कहानियाँ सुनाता है।",
    "हमारे गाँव में बिजली नहीं आती रात को।",
    "उसने अपनी परीक्षा की तैयारी पूरी कर ली।",
    "यह फोटो बहुत सुंदर आई है।",
    "माँ ने आज आलू का पराठा बनाया।",
    "वह अपने भाई के साथ खेल रहा है।",
    "हम लोग हर साल होली खेलते हैं।",
    "उसने अपनी किताब गायब कर दी।",
    "मुझे क्रिकेट देखना बहुत पसंद है।",
    "वह बहुत शांतिप्रिय लड़का है।",
    "हमने बाजार से नए कपड़े खरीदे।",
    "उसके पास बहुत सारे पेन हैं।",
    "आज स्कूल में बहुत मज़ा आया।",
    "मेरी दादी रोज़ कहानियाँ सुनाती हैं।",
    "वह अपने पिता के साथ दुकान जाता है।",
    "हम सब मिलकर नाच रहे थे।",
    "उसने नई साइकिल माँगी है।",
    "स्कूल में आज ड्राइंग कॉम्पिटिशन है।",
    "मुझे आइसक्रीम बहुत पसंद है।",
    "वह बहुत अच्छी कविता सुनाता है।",
    "हमारे मोहल्ले में नया पार्क बना है।",
    "उसने अपनी कॉपी साफ-सुथरी रखी है।",
    "माँ ने सबको गरमागरम समोसे दिए।",
    "वह अपने दोस्तों के साथ मस्ती करता है।",
    "हम लोग हर रविवार को मंदिर जाते हैं।",
    "उसने अपनी पुरानी किताब बेच दी।",
    "यह कहानी बहुत प्रेरणादायक है।",
    "मुझे हरा रंग बहुत पसंद है।",
    "वह बहुत तेज़ लिखता है।",
    "हमने पिछले हफ्ते पार्टी की थी।",
    "उसके पास बहुत सारे खिलौने हैं।",
    "आज का दिन बहुत खुशनुमा है।",
    "मेरी नानी बहुत अच्छी रोटी बनाती हैं।",
    "वह अपने चाचा के साथ बाजार गया।",
    "हम सब मिलकर गाना गाते हैं।",
    "उसने नया बैग माँगा है।",
    "स्कूल में आज कविता पाठ होगा।",
    "मुझे चॉकलेट बहुत पसंद है।",
    "वह बहुत अच्छी कहानी लिखता है।",
    "हमारे गाँव में बहुत सारे फूल हैं।",
    "उसने अपनी पेन खो दी है।",
    "माँ ने आज खीर बनाई है।",
    "वह अपने भाई-बहन के साथ खेलता है।",
    "हम लोग हर साल दिवाली मनाते हैं।",
    "उसने अपनी डायरी में चित्र बनाया।",
    "मुझे फुटबॉल खेलना अच्छा लगता है।",
    "वह बहुत शांत और समझदार है।",
    "हमने बाजार से फल खरीदे।",
    "उसके पास बहुत सारे रंगीन पेन हैं।",
    "आज स्कूल में बहुत अच्छा लगा।",
    "मेरी मम्मी बहुत अच्छी साड़ी पहनती हैं।",
    "वह अपने पापा के साथ सैर पर गया।",
    "हम सब मिलकर हँस रहे थे।",
    "उसने नई जूती माँगी है।",
    "स्कूल में आज नया नियम लगा है।",
    "मुझे मिठाई बहुत पसंद है।",
    "वह बहुत अच्छी डांस करती है।",
    "हमारे घर के पास एक मंदिर है।",
    "उसने अपनी किताबें व्यवस्थित रखी हैं।",
    "माँ ने सबको गरम चाय दी।",
    "वह अपने दोस्त के साथ साइकिल चलाता है।",
    "हम लोग हर शाम गपशप करते हैं।",
    "उसने अपनी पुरानी साइकिल ठीक कराई।",
    "यह किताब बहुत रोचक है।",
    "मुझे पीला रंग पसंद है।",
    "वह बहुत तेज़ पढ़ता है।",
    "हमने रविवार को मज़े किए।",
    "उसके पास बहुत सारे कार्टून हैं।",
    "आज का मौसम बहुत ठंडा है।",
    "मेरी मौसी बहुत अच्छी बातें करती हैं।",
    "वह अपने मामा के साथ घूमने गया।",
    "हम सब मिलकर खाना खा रहे हैं।",
    "उसने नया रंग माँगा है।",
    "स्कूल में आज कहानी सुनाई जाएगी।",
    "मुझे बिरयानी बहुत पसंद है।",
    "वह बहुत अच्छी कविता पढ़ता है।",
    "हमारे मोहल्ले में बहुत बच्चे हैं।",
    "उसने अपनी पेंसिल खो दी।",
    "माँ ने आज हलवा बनाया है।",
    "वह अपने दोस्तों के साथ मस्ती करता है।",
    "हम लोग हर साल छठ पूजा मनाते हैं।",
    "उसने अपनी नई डायरी शुरू की।",
    "मुझे बैडमिंटन खेलना अच्छा लगता है।",
    "वह बहुत मेहनती छात्र है।",
    "हमने बाजार से कपड़े खरीदे।",
    "उसके पास बहुत सारे किताबें हैं।",
    "आज स्कूल में छुट्टी है।",
    "मेरी बुआ बहुत अच्छी खाना बनाती हैं।",
    "वह अपने भाई के साथ क्रिकेट खेलता है।",
    "हम सब मिलकर गप्पें मार रहे थे।",
    "उसने नया लैपटॉप माँगा है।",
    "स्कूल में आज क्विज होगा।",
    "मुझे पिज़्ज़ा बहुत पसंद है।",
    "वह बहुत अच्छी पेंटिंग बनाती है।",
    "हमारे गाँव में बहुत शांति है।",
    "उसने अपनी कॉपी चेक कराई।",
    "माँ ने सबको गरम-गरम पूरी दी।",
    "वह अपने पिता के साथ दुकान जाता है।",
    "हम लोग हर रविवार को मूवी देखते हैं।",
    "उसने अपनी पुरानी टी-शर्ट फेंक दी।",
    "यह कहानी बहुत प्यारी है।",
    "मुझे लाल रंग बहुत पसंद है।",
    "वह बहुत तेज़ दौड़ लगाता है।",
    "हमने पिछले महीने घूमने गए थे।",
    "उसके पास बहुत सारे खिलौने हैं।",
    "आज का दिन बहुत व्यस्त बीता।",
    "मेरी मम्मी रोज़ सुबह जल्दी उठती हैं।",
    "वह अपने दोस्तों के साथ मज़े करता है।",
    "हम सब मिलकर हँसी-मज़ाक कर रहे थे।",
    "उसने नई वॉटर बॉटल माँगी है।",
    "स्कूल में आज स्पोर्ट्स डे है।",
    "मुझे समोसा बहुत पसंद है।",
    "वह बहुत अच्छी कहानी सुनाता है।",
    "हमारे घर के पास एक स्कूल है।",
    "उसने अपनी किताबें पैक कर ली हैं।",
    "माँ ने आज चावल-दाल बनाई है।",
    "वह अपने भाई-बहन के साथ खेलता है।",
    "हम लोग हर साल रक्षाबंधन मनाते हैं।",
    "उसने अपनी नई पेन से लिखा।",
    "मुझे टेनिस खेलना अच्छा लगता है।",
    "वह बहुत ईमानदार बच्चा है।",
    "हमने बाजार से फल-सब्ज़ियाँ खरीदीं।",
    "उसके पास बहुत सारे रंग हैं।",
    "आज स्कूल में बहुत भीड़ थी।",
    "मेरी नानी बहुत अच्छी लोरी सुनाती हैं।",
    "वह अपने चाचा के साथ सैर करता है।",
    "हम सब मिलकर खाना बना रहे हैं।",
    "उसने नया रबड़ माँगा है।",
    "स्कूल में आज नाटक होगा।",
    "मुझे बर्गर बहुत पसंद है।",
    "वह बहुत अच्छी कविता लिखता है।",
    "हमारे मोहल्ले में बहुत हरियाली है।",
    "उसने अपनी पेंसिलें व्यवस्थित रखी हैं।",
    "माँ ने सबको गरमागरम हलवा दिया।",
    "वह अपने दोस्तों के साथ साइकिल चलाता है।",
    "हम लोग हर शाम चाय पीते हैं।",
    "उसने अपनी पुरानी जूती ठीक कराई।",
    "यह किताब बहुत ज्ञानवर्धक है।",
    "मुझे नीला रंग बहुत पसंद है।",
    "वह बहुत तेज़ गाना गाता है।",
    "हमने रविवार को पिकनिक मनाई।",
    "उसके पास बहुत सारे कार्ड हैं।",
    "आज का मौसम बहुत सुंदर है।",
    "मेरी मौसी बहुत अच्छी सलाह देती हैं।",
    "वह अपने मामा के साथ बाजार गया।",
    "हम सब मिलकर पार्टी कर रहे हैं।",
    "उसने नया स्केच पेन माँगा है।",
    "स्कूल में आज कहानी प्रतियोगिता है।",
    "मुझे पास्ता बहुत पसंद है।",
    "वह बहुत अच्छी ड्राइंग करता है।",
    "हमारे गाँव में बहुत शांति है।",
    "उसने अपनी कॉपी बहुत साफ रखी है।",
    "माँ ने आज पकौड़े बनाए हैं।",
    "वह अपने भाई के साथ मस्ती करता है।",
    "हम लोग हर साल नवरात्रि मनाते हैं।",
    "उसने अपनी नई डायरी में लिखा।",
    "मुझे वॉलीबॉल खेलना अच्छा लगता है।",
    "वह बहुत मेहनती और समझदार है।",
    "हमने बाजार से नए जूते खरीदे।",
    "उसके पास बहुत सारी किताबें हैं।",
    "आज स्कूल में बहुत मज़ा आया।",
    "मेरी बुआ बहुत अच्छी कहानियाँ सुनाती हैं।",
    "वह अपने पापा के साथ घूमने गया।",
    "हम सब मिलकर गाना गा रहे हैं।",
    "उसने नया स्कूल बैग माँगा है।",
    "स्कूल में आज कविता पाठ होगा।",
    "मुझे आइसक्रीम बहुत पसंद है।",
    "वह बहुत अच्छी पेंटिंग बनाती है।",
    "हमारे घर के पास एक बड़ा पार्क है।",
    "उसने अपनी किताबें ठीक से रखी हैं।",
    "माँ ने सबको गरम-गरम चाय दी।",
    "वह अपने दोस्तों के साथ क्रिकेट खेलता है।",
    "हम लोग हर रविवार को मंदिर जाते हैं।",
    "उसने अपनी पुरानी साइकिल बेच दी।",
    "यह कहानी बहुत दिलचस्प है।",
    "मुझे हरा रंग बहुत पसंद है।",
    "वह बहुत तेज़ पढ़ाई करता है।",
    "हमने पिछले हफ्ते मज़े किए।",
    "उसके पास बहुत सारे खिलौने हैं।",
    "आज का दिन बहुत अच्छा बीता।",
    "मेरी मम्मी रोज़ सुबह जल्दी उठती हैं।",
    "वह अपने दोस्तों के साथ हँसता-खेलता है।",
    "हम सब मिलकर मस्ती कर रहे थे।",
    "उसने नई वॉटर बॉटल ली है।",
    "स्कूल में आज स्पोर्ट्स डे मनाया जाएगा।",
    "मुझे समोसा और चाय बहुत पसंद है।",
    "वह बहुत अच्छी कहानी सुनाता है।",
    "हमारे घर के पास एक मंदिर है।",
    "उसने अपनी कॉपी बहुत साफ-सुथरी रखी है।",
    "माँ ने आज चावल-राजमा बनाया है।",
    "वह अपने भाई-बहन के साथ खेलता रहता है।",
    "हम लोग हर साल गणतंत्र दिवस मनाते हैं।",
    "उसने अपनी नई पेन से नोट्स बनाए।",
    "मुझे क्रिकेट बहुत पसंद है।",
    "वह बहुत ईमानदार और मेहनती है।",
    "हमने बाजार से ताज़े फल खरीदे।",
    "उसके पास बहुत सारे रंगीन पेन हैं।",
    "आज स्कूल में छुट्टी लग गई।",
    "मेरी नानी बहुत प्यार से बात करती हैं।",
    "वह अपने चाचा के साथ सैर करने गया।",
    "हम सब मिलकर खाना बना रहे हैं।",
    "उसने नया रबड़ और पेन माँगा है।",
    "स्कूल में आज नाटक का अभ्यास होगा।",
    "मुझे पिज़्ज़ा और बर्गर पसंद हैं।",
    "वह बहुत अच्छी कविता लिखती है।",
    "हमारे मोहल्ले में बहुत हरियाली है।",
    "उसने अपनी पेंसिलें सही जगह रखी हैं।",
    "माँ ने सबको गरमागरम हलवा परोसा।",
    "वह अपने दोस्तों के साथ साइकिल चलाता है।",
    "हम लोग हर शाम साथ बैठकर बातें करते हैं।",
    "उसने अपनी पुरानी जूती बदल दी।",
    "यह किताब बहुत रोचक और ज्ञानवर्धक है।",
    "मुझे नीला और हरा दोनों रंग पसंद हैं।",
    "वह बहुत तेज़ गाना गाता है।",
    "हमने रविवार को परिवार के साथ समय बिताया।",
    "उसके पास बहुत सारे कार्टून बुक हैं।",
    "आज का मौसम बहुत सुहावना है।",
    "मेरी मौसी बहुत अच्छी सलाह देती हैं।",
    "वह अपने मामा के साथ बाजार गया है।",
    "हम सब मिलकर जन्मदिन मना रहे हैं।",
    "उसने नया स्केच पेन और रंग माँगा है।",
    "स्कूल में आज कहानी सुनाने का कार्यक्रम है।",
    "मुझे पास्ता और नूडल्स बहुत पसंद हैं।",
    "वह बहुत अच्छी ड्राइंग और पेंटिंग करता है।",
    "हमारे गाँव में बहुत सारी शांति है।",
    "उसने अपनी कॉपी बहुत सुंदर रखी है।",
    "माँ ने आज पकौड़े और चटनी बनाई है।",
    "वह अपने भाई के साथ रोज़ मस्ती करता है।",
    "हम लोग हर साल दशहरा और दीवाली मनाते हैं।",
    "उसने अपनी नई डायरी में सपने लिखे हैं।",
    "मुझे वॉलीबॉल और बैडमिंटन दोनों पसंद हैं।",
    "वह बहुत मेहनती और समझदार बच्चा है।",
    "हमने बाजार से नए कपड़े और जूते खरीदे।",
    "उसके पास बहुत सारी किताबें और कॉमिक्स हैं।",
    "आज स्कूल में बहुत अच्छा माहौल था।",
    "मेरी बुआ बहुत प्यार से खाना खिलाती हैं।",
    "वह अपने पापा के साथ पार्क गया है।",
    "हम सब मिलकर गाना और नाच रहे हैं।",
    "उसने नया स्कूल बैग और बॉटल माँगा है।",
    "स्कूल में आज कविता और भाषण प्रतियोगिता है।",
    "मुझे आइसक्रीम और चॉकलेट दोनों चाहिए।",
    "वह बहुत अच्छी पेंटिंग और क्राफ्ट बनाती है।",
    "हमारे घर के पास एक बहुत सुंदर पार्क है।",
    "उसने अपनी सारी किताबें ठीक से पैक की हैं।",
    "माँ ने सबको गरम-गरम पराठे और अचार दिए।",
    "वह अपने दोस्तों के साथ रोज़ क्रिकेट खेलता है।",
    "हम लोग हर रविवार को परिवार के साथ घूमने जाते हैं।",
    "उसने अपनी पुरानी साइकिल को रंगवाया है।",
    "यह कहानी बहुत प्यारी और सीख देने वाली है।",
    "मुझे लाल, नीला और हरा रंग पसंद हैं।",
    "वह बहुत तेज़ दौड़ता और कूदता है।",
    "हमने पिछले महीने समुद्र तट पर घूमने गए थे।",
    "उसके पास बहुत सारे खिलौने और गेम्स हैं।",
    "आज का दिन बहुत खुशहाल बीता है।",
    "मेरी मम्मी रोज़ सुबह सबके लिए नाश्ता बनाती हैं।",
    "वह अपने दोस्तों के साथ बहुत मज़े करता है।",
    "हम सब मिलकर हँसी-मज़ाक और खेल रहे थे।",
    "उसने नई वॉटर बॉटल और लंच बॉक्स लिया है।",
    "स्कूल में आज खेलकूद का दिन है।",
    "मुझे समोसा, पकौड़ा और जलेबी पसंद हैं।",
    "वह बहुत अच्छी कहानियाँ और कविताएँ सुनाता है।",
    "हमारे घर के पास एक बड़ा और साफ तालाब है।",
    "उसने अपनी कॉपी और किताबें बहुत साफ रखी हैं।",
    "माँ ने आज चावल, दाल और सब्ज़ी बनाई है।",
    "वह अपने भाई-बहन के साथ रोज़ खेलता-कूदता है।",
    "हम लोग हर साल स्वतंत्रता दिवस मनाते हैं।",
    "उसने अपनी नई पेन से बहुत सुंदर लिखा है।",
    "मुझे क्रिकेट, फुटबॉल और टेनिस पसंद हैं।",
    "वह बहुत ईमानदार, मेहनती और समझदार है।",
    "हमने बाजार से ताज़े फल, सब्ज़ियाँ और मिठाई खरीदी।",
    "उसके पास बहुत सारे रंगीन पेन, पेंसिल और क्रेयॉन हैं।",
    "आज स्कूल में छुट्टी होने से बहुत खुशी हुई।",
    "मेरी नानी बहुत प्यार से लोरी और कहानियाँ सुनाती हैं।",
    "वह अपने चाचा के साथ रोज़ सैर करने जाता है।",
    "हम सब मिलकर खाना बना रहे हैं और मज़े ले रहे हैं।",
    "उसने नया रबड़, स्केल और ज्योमेट्री बॉक्स माँगा है।",
    "स्कूल में आज नाटक और नृत्य का कार्यक्रम है।",
    "मुझे पिज़्ज़ा, बर्गर और पास्ता बहुत पसंद हैं।",
    "वह बहुत अच्छी कविताएँ लिखता और सुनाता है।",
    "हमारे मोहल्ले में बहुत हरियाली और शांति है।",
    "उसने अपनी सारी पेंसिलें और रंग ठीक से रखे हैं।",
    "माँ ने सबको गरमागरम हलवा और पूरी परोसी है।",
    "वह अपने दोस्तों के साथ साइकिल चलाकर बहुत मज़े करता है।",
    "हम लोग हर शाम साथ बैठकर बातें करते और हँसते हैं।",
    "उसने अपनी पुरानी जूती को नए जूतों से बदल दिया है।",
    "यह किताब बहुत रोचक, ज्ञानवर्धक और प्रेरणादायक है।",
    "मुझे नीला, हरा और पीला रंग बहुत पसंद हैं।",
    "वह बहुत तेज़ गाना गाता और नाचता है।",
    "हमने रविवार को पूरे परिवार के साथ पिकनिक मनाई।",
    "उसके पास बहुत सारे कार्टून, कॉमिक्स और कहानी की किताबें हैं।",
    "आज का मौसम बहुत सुंदर और सुहावना है।",
    "मेरी मौसी बहुत अच्छी सलाह और मदद करती हैं।",
    "वह अपने मामा के साथ बाजार और पार्क जाता है।",
    "हम सब मिलकर जन्मदिन मना रहे हैं और केक काट रहे हैं।",
    "उसने नया स्केच पेन, रंग और ड्राइंग बुक माँगा है।",
    "स्कूल में आज कहानी सुनाने और कविता पढ़ने का दिन है।",
    "मुझे पास्ता, नूडल्स और मैगी बहुत पसंद हैं।",
    "वह बहुत अच्छी ड्राइंग, पेंटिंग और क्राफ्ट बनाता है।",
    "हमारे गाँव में बहुत सारी हरियाली और शांति है।",
    "उसने अपनी सारी कॉपियाँ और किताबें बहुत सुंदर रखी हैं।",
    "माँ ने आज पकौड़े, चटनी और गरम चाय बनाई है।",
    "वह अपने भाई के साथ रोज़ मस्ती और खेल करता है।",
    "हम लोग हर साल दशहरा, दीवाली और होली बड़े धूमधाम से मनाते हैं।",
    "उसने अपनी नई डायरी में सपने, विचार और चित्र बनाए हैं।",
    "मुझे वॉलीबॉल, बैडमिंटन और टेबल टेनिस खेलना पसंद है।",
    "वह बहुत मेहनती, ईमानदार और समझदार छात्र है।",
    "हमने बाजार से नए कपड़े, जूते और बैग खरीदे हैं।",
    "उसके पास बहुत सारी किताबें, कॉमिक्स और स्टोरी बुक हैं।",
    "आज स्कूल में बहुत अच्छा माहौल और मज़ा था।",
    "मेरी बुआ बहुत प्यार से खाना खिलाती और बातें करती हैं।",
    "वह अपने पापा के साथ पार्क, बाजार और सैर पर जाता है।",
    "हम सब मिलकर गाना, नाचना और मस्ती कर रहे हैं।",
    "उसने नया स्कूल बैग, वॉटर बॉटल और लंच बॉक्स लिया है।",
    "स्कूल में आज कविता, भाषण और कहानी प्रतियोगिता है।",
    "मुझे आइसक्रीम, चॉकलेट, केक और मिठाई बहुत पसंद हैं।",
    "वह बहुत अच्छी पेंटिंग, क्राफ्ट और ड्राइंग बनाती है।",
    "हमारे घर के पास एक बहुत सुंदर, साफ और हरा-भरा पार्क है।"
]

def generate_massive_dataset(clean_sentences, num_samples=10000):
    """Generates N synthetic test cases using Phase 3 logic."""
    print(f"Generating {num_samples:,} synthetic test cases...")
    dataset = []
    
    for _ in range(num_samples):
        # 1. Pick a random clean sentence
        clean_text = random.choice(clean_sentences)
        words = clean_text.split()
        
        # 2. Pick a random word to corrupt (skipping tiny words)
        valid_indices = [i for i, w in enumerate(words) if len(w) > 2]
        
        if not valid_indices:
            dataset.append({"typo": clean_text, "clean": clean_text})
            continue
            
        target_idx = random.choice(valid_indices)
        original_word = words[target_idx]
        
        # 3. Apply the Phase 3 typo generator
        # (Assumes generate_hindi_typo from Phase 3 is in memory)
        typo_word = generate_hindi_typo(original_word)
        
        # 4. Reconstruct the broken sentence
        typo_words = words.copy()
        typo_words[target_idx] = typo_word
        typo_text = " ".join(typo_words)
        
        dataset.append({"typo": typo_text, "clean": clean_text})
        
    print("Dataset generation complete.\n")
    return dataset

def evaluate_massive_spell_checker(test_dataset):
    """Evaluates the pipeline and calculates Precision, Recall, and F1."""
    print(f"Starting pipeline evaluation for {len(test_dataset):,} sentences...")
    
    total_words = 0
    true_positives = 0  
    false_positives = 0 
    false_negatives = 0 
    true_negatives = 0  
    
    start_time = time.time()
    
    # We wrap the loop in tqdm for a clean progress bar
    for data in tqdm(test_dataset, desc="Processing Sentences", unit="sent"):
        typo_text = data["typo"]
        ground_truth_text = data["clean"]
        
        # Run through Phase 4 pipeline
        predicted_text, _ = correct_hindi_sentence(typo_text, min_freq=5000)
        
        typo_words = typo_text.split()
        truth_words = ground_truth_text.split()
        pred_words = predicted_text.split()
        
        if not (len(typo_words) == len(truth_words) == len(pred_words)):
            continue
            
        for t_word, truth_word, p_word in zip(typo_words, truth_words, pred_words):
            total_words += 1
            is_typo = (t_word != truth_word)
            model_changed_word = (t_word != p_word)
            
            if is_typo:
                if model_changed_word and p_word == truth_word:
                    true_positives += 1
                else:
                    false_negatives += 1
            else:
                if model_changed_word:
                    false_positives += 1
                else:
                    true_negatives += 1

    end_time = time.time()
    total_time = end_time - start_time
    
    # Calculate Core Metrics
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (true_positives + true_negatives) / total_words if total_words > 0 else 0
    
    # Print Final Report
    print("\n" + "=" * 50)
    print("FINAL PERFORMANCE REPORT")
    print("=" * 50)
    print(f"Total Sentences Processed : {len(test_dataset):,}")
    print(f"Total Words Analyzed      : {total_words:,}")
    print(f"Total Processing Time     : {total_time:.2f} seconds")
    print(f"Average Time per Sentence : {(total_time/len(test_dataset)):.4f} seconds\n")
    
    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision * 100:.2f}%")
    print(f"Recall    : {recall * 100:.2f}%")
    print(f"F1-Score  : {f1_score * 100:.2f}%\n")
    
    print("DETAILED BREAKDOWN:")
    print(f"True Positives (Correctly Fixed)   : {true_positives:,}")
    print(f"True Negatives (Correctly Ignored) : {true_negatives:,}")
    print(f"False Positives (Wrongly Changed)  : {false_positives:,}")
    print(f"False Negatives (Missed Typos)     : {false_negatives:,}")
    print("=" * 50)


massive_dataset = generate_massive_dataset(BASE_CLEAN_SENTENCES, num_samples=10000)
evaluate_massive_spell_checker(massive_dataset)